<a href="https://colab.research.google.com/github/leandrojgarridol-ui/Tarea1_IEE2714/blob/main/Tarea1_IEE2714.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from ipywidgets import interact, FloatSlider

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!unzip "/content/ArchivosT1.zip"

In [ ]:
P1= cv2.imread("/content/P1_IMG_2402.tif")
cv2_imshow(P1)

In [ ]:
P1BGR= P1[:,:,::-1]
cv2_imshow(P1BGR)

Antes de poder definir el modelo que quiero usar, decidí utilizar los códigos que previamente se nos mostraron en la cápsula 2 del curso, utilizando las funciones para modificar y ver bien cuál de los 3 modos HS me gusta más para hacer la herramienta.

In [ ]:
def rgb_to_hsi(img: np.ndarray) -> np.ndarray:
  img = img.astype(np.float64)
  img = img / 255

  R = img[:,:, 0]
  G = img[:,:, 1]
  B = img[:,:, 2]

  I = (R + G + B) / 3.0
  epsilon = 1e-10

  min_rgb = np.minimum(np.minimum(R, G), B)
  max_rgb = np.maximum(np.maximum(R, G), B)
  delta = max_rgb - min_rgb
  S = 1.0 -(3.0 / (R + G + B + epsilon)) * min_rgb

  num_theta = 0.5 * ((R - G) + (R - B))
  den_theta = np.sqrt((R -G) ** 2 + (R - B) * (G - B)) + epsilon
  theta = np.arccos(np.clip(num_theta / den_theta, -1.0, 1.0))

  H = np.degrees(theta)
  H = np.where(B > G, 360.0 - H, H)

  H = np.where(delta < epsilon, 0.0, H)

  hsi_img = np.stack([H, S, I], axis=-1)
  return hsi_img


def plot_hsi(hsi_img: np.ndarray):

  H = hsi_img[:, :, 0]
  S = hsi_img[:, :, 1]
  I = hsi_img[:, :, 2]

  fig, axes = plt.subplots(1, 3, figsize=(15, 5))

  canales = [
      (H, "Hue (H)", "hsv", 0, 360),
      (S, "Saturation (S)", "Reds", 0, 1),
      (I, "Intensity (I)", "gray", 0, 1),
  ]

  for ax, (canal, titulo, cmap, vmin, vmax) in zip(axes, canales):
      im = ax.imshow(canal, cmap=cmap, vmin=vmin, vmax=vmax)
      ax.set_title(titulo)
      ax.axis("off")
      fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

  fig.tight_layout()

In [ ]:
def rgb_to_hsv(img: np.ndarray) -> np.ndarray:

  img = img.astype(np.float64)
  img = img/ 255.0

  R = img[:, :, 0]
  G = img[:, :, 1]
  B = img[:, :, 2]

  epsilon = 1e-10
  C_max = np.maximum(np.maximum(R, G), B)
  C_min = np.minimum(np.minimum(R, G), B)
  delta = C_max - C_min


  V = C_max
  S = np.where(C_max > epsilon, delta / (V + epsilon), 0.0)
  H = np.zeros_like(V)

  mask_r = (C_max == R) * (delta > epsilon)
  mask_g = (C_max == G) * (delta > epsilon)
  mask_b = (C_max == B) * (delta > epsilon)

  H[mask_r] = 60.0 * (((G[mask_r] - B[mask_r]) / delta[mask_r]) % 6)
  H[mask_g] = 60.0 * (((B[mask_g] - R[mask_g]) / delta[mask_g]) + 2)
  H[mask_b]= 60.0 * (((R[mask_b] - G[mask_b]) / delta[mask_b]) + 4)

  H = np.where(H < 0, H + 360.0, H)
  H = np.where(delta < epsilon, 0.0, H)
  hsv_img = np.stack([H, S, V], axis=-1)
  return hsv_img


def plot_hsv(hsv_img: np.ndarray):

  H = hsv_img[:, :, 0]
  S = hsv_img[:, :, 1]
  V = hsv_img[:, :, 2]

  fig, axes = plt.subplots(1, 3, figsize=(15, 5))

  canales = [
      (H, "Hue (H)", "hsv", 0, 360),
      (S, "Saturation (S)", "Reds", 0, 1),
      (V, "Value (V)", "gray", 0, 1)
  ]

  for ax, (canal, titulo, cmap, vmin, vmax) in zip(axes, canales):
    im = ax.imshow(canal, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(titulo)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

  fig.tight_layout()

In [ ]:
def rgb_to_hsl(img: np.ndarray) -> np.ndarray:

  img = img.astype(np.float64)
  img = img/ 255.0

  R = img[:, :, 0]
  G = img[:, :, 1]
  B = img[:, :, 2]
  epsilon = 1e-10

  C_max = np.maximum(np.maximum(R, G), B)
  C_min = np.minimum(np.minimum(R, G), B)
  delta = C_max - C_min

  L = (C_max + C_min) / 2.0
  den = 1.0 - np.abs(2.0 * L -1.0)
  S = np.where(delta > epsilon, delta / (den + epsilon), 0.0)
  H = np.zeros_like(C_max)

  mask_r = (C_max == R) * (delta > epsilon)
  mask_g = (C_max == G) * (delta > epsilon)
  mask_b = (C_max == B) * (delta > epsilon)

  H[mask_r] = 60.0 * (((G[mask_r] - B[mask_r]) / delta[mask_r]) % 6)
  H[mask_g] = 60.0 * (((B[mask_g] - R[mask_g]) / delta[mask_g]) + 2)
  H[mask_b]= 60.0 * (((R[mask_b] - G[mask_b]) / delta[mask_b]) + 4)

  H = np.where(H < 0, H + 360.0, H)
  H = np.where(delta < epsilon, 0.0, H)
  hsl_img = np.stack([H, S, L], axis=-1)
  return hsl_img


def plot_hsl(hsl_img: np.ndarray):

  H = hsl_img[:, :, 0]
  S = hsl_img[:, :, 1]
  L = hsl_img[:, :, 2]

  fig, axes = plt.subplots(1, 3, figsize=(15, 5))

  canales = [
      (H, "Hue (H)", "hsv", 0, 360),
      (S, "Saturation (S)", "Reds", 0, 1),
      (L, "Lightness (L)", "gray", 0, 1)
  ]

  for ax, (canal, titulo, cmap, vmin, vmax) in zip(axes, canales):
    im = ax.imshow(canal, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(titulo)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

  fig.tight_layout()

In [ ]:
hsi_P1= rgb_to_hsi(P1)
plot_hsi(hsi_P1)

In [ ]:
hsv_p1= rgb_to_hsv(P1)
plot_hsv(hsv_p1)

In [ ]:
hsl_p1= rgb_to_hsl(P1BGR)
plot_hsl(hsl_p1)

In [ ]:
def hsi_to_rgb(hsi_img: np.ndarray) -> np.ndarray:

  H = hsi_img[:, :, 0] % 360.0
  S = np.clip(hsi_img[:, :, 1], 0.0, 1.0)
  I = np.clip(hsi_img[:, :, 2], 0.0, 1.0)

  epsilon = 1e-10
  Hr = np.radians(H)

  R = np.zeros_like(I)
  G = np.zeros_like(I)
  B = np.zeros_like(I)

  # Sector RG: 0 <= H < 120
  m = (H >= 0) & (H < 120)
  B[m] = I[m] * (1 - S[m])
  R[m] = I[m] * (1 + (S[m] * np.cos(Hr[m])) / (np.cos(np.radians(60) - Hr[m]) + epsilon))
  G[m] = 3 * I[m] - (R[m] + B[m])

  # Sector GB: 120 <= H < 240
  m = (H >= 120) & (H < 240)
  Hp = Hr[m] - np.radians(120)
  R[m] = I[m] * (1 - S[m])
  G[m] = I[m] * (1 + (S[m] * np.cos(Hp)) / (np.cos(np.radians(60) - Hp) + epsilon))
  B[m] = 3 * I[m] - (R[m] + G[m])

  # Sector BR: 240 <= H <= 360
  m = (H >= 240) & (H <= 360)
  Hp = Hr[m] - np.radians(240)
  G[m] = I[m] * (1 - S[m])
  B[m] = I[m] * (1 + (S[m] * np.cos(Hp)) / (np.cos(np.radians(60) - Hp) + epsilon))
  R[m] = 3 * I[m] - (G[m] + B[m])

  rgb_img = np.clip(np.stack([R, G, B], axis=-1), 0.0, 1.0)
  return rgb_img


def hsv_to_rgb(hsv_img: np.ndarray) -> np.ndarray:

  H = hsv_img[:, :, 0] % 360.0
  S = np.clip(hsv_img[:, :, 1], 0.0, 1.0)
  V = np.clip(hsv_img[:, :, 2], 0.0, 1.0)

  C = V * S
  Hp = H / 60.0
  X = C * (1 - np.abs(Hp % 2 - 1))
  m = V - C

  Rp = np.zeros_like(V)
  Gp = np.zeros_like(V)
  Bp = np.zeros_like(V)

  mask0 = (Hp >= 0) & (Hp < 1)
  mask1 = (Hp >= 1) & (Hp < 2)
  mask2 = (Hp >= 2) & (Hp < 3)
  mask3 = (Hp >= 3) & (Hp < 4)
  mask4 = (Hp >= 4) & (Hp < 5)
  mask5 = (Hp >= 5) & (Hp <= 6)

  Rp[mask0], Gp[mask0], Bp[mask0] = C[mask0], X[mask0], 0.0
  Rp[mask1], Gp[mask1], Bp[mask1] = X[mask1], C[mask1], 0.0
  Rp[mask2], Gp[mask2], Bp[mask2] = 0.0, C[mask2], X[mask2]
  Rp[mask3], Gp[mask3], Bp[mask3] = 0.0, X[mask3], C[mask3]
  Rp[mask4], Gp[mask4], Bp[mask4] = X[mask4], 0.0, C[mask4]
  Rp[mask5], Gp[mask5], Bp[mask5] = C[mask5], 0.0, X[mask5]

  rgb_img = np.clip(np.stack([Rp + m, Gp + m, Bp + m], axis=-1), 0.0, 1.0)
  return rgb_img


def hsl_to_rgb(hsl_img: np.ndarray) -> np.ndarray:

  H = hsl_img[:, :, 0] % 360.0
  S = np.clip(hsl_img[:, :, 1], 0.0, 1.0)
  L = np.clip(hsl_img[:, :, 2], 0.0, 1.0)

  C = (1 - np.abs(2 * L - 1)) * S
  Hp = H / 60.0
  X = C * (1 - np.abs(Hp % 2 - 1))
  m = L - C / 2.0

  Rp = np.zeros_like(L)
  Gp = np.zeros_like(L)
  Bp = np.zeros_like(L)

  mask0 = (Hp >= 0) & (Hp < 1)
  mask1 = (Hp >= 1) & (Hp < 2)
  mask2 = (Hp >= 2) & (Hp < 3)
  mask3 = (Hp >= 3) & (Hp < 4)
  mask4 = (Hp >= 4) & (Hp < 5)
  mask5 = (Hp >= 5) & (Hp <= 6)

  Rp[mask0], Gp[mask0], Bp[mask0] = C[mask0], X[mask0], 0.0
  Rp[mask1], Gp[mask1], Bp[mask1] = X[mask1], C[mask1], 0.0
  Rp[mask2], Gp[mask2], Bp[mask2] = 0.0, C[mask2], X[mask2]
  Rp[mask3], Gp[mask3], Bp[mask3] = 0.0, X[mask3], C[mask3]
  Rp[mask4], Gp[mask4], Bp[mask4] = X[mask4], 0.0, C[mask4]
  Rp[mask5], Gp[mask5], Bp[mask5] = C[mask5], 0.0, X[mask5]

  rgb_img = np.clip(np.stack([Rp + m, Gp + m, Bp + m], axis=-1), 0.0, 1.0)
  return rgb_img

In [ ]:
def tune_cylindrical(rgb_img: np.ndarray, to_func, from_func, third_label: str, third_default: float = 1.0):

    base = to_func(rgb_img)

    #Función auxiliar para obtener los parámetros del espacio
    def _update(delta_h = 0.0, scale_s = 1.0, scale_third = third_default):

        tuned = base.copy() #Copia para no afectar img original
        tuned[:, :, 0] = (tuned[:, :, 0] + delta_h) % 360.0 #Obtención del ángulo
        tuned[:, :, 1] = np.clip(tuned[:, :, 1] *scale_s, 0.0, 1.0) #Componente de saturación
        tuned[:, :, 2] = np.clip(tuned[:, :, 2] * scale_third, 0.0, 1.0) #Componente de brillo

        rgb_out = from_func(tuned)

        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_out)
        plt.axis("off")
        plt.show()

    interact(
        _update,
        delta_h = FloatSlider(min=-180, max=180, step=5, value=0.0, description="Delta H"),
        scale_s = FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0, description="Escala S"),
        scale_third= FloatSlider(min=0.0, max=2.0, step=0.05, value=third_default,
                                 description=f"Escala {third_label}"),
    )

Cómo solo debo jugar con el parametro de la saturación, solo jugué con este para poder ver bien cómo cambiaban los colores de la imagen, quedandome con la que más me gustará antes de decidir. Nuevamente, todas estas funciones fueron previamente vistas en las cápsulas pasadas del curso.

In [ ]:
tune_cylindrical(P1BGR, rgb_to_hsi, hsi_to_rgb, third_label="I")

In [ ]:
tune_cylindrical(P1BGR, rgb_to_hsv, hsv_to_rgb, third_label="V")

In [ ]:
tune_cylindrical(P1BGR, rgb_to_hsl, hsl_to_rgb, third_label="L")

Me voy a quedar con HSL, porque en general, no quiero que al modificar la saturación, la imagen sea afectada por la luminosidad. Así que, estéticamente creo que favorece. Así que ahora voy a tratar de determinar puntos y el factor de saturación para cada color. Según yo, para esta imagen me gustaría saturar más los colores azules y amarillos, más moderadamente los verdes, y tratando de saturar menos los rojos, ya que, creo que lo que importa destacar más al pájaro central de la foto. Saturando menos blancos, negros y grises para no perder detalles, ni destacar las sombras.
<br>
<br>


In [ ]:
# Primero se va a definir la linealización gamma canal a canal
# mediante esta función auxiliar

def _gamma_to_linear(c: np.ndarray, gamma: float = 2.2) -> np.ndarray:
  return c ** gamma

def rgb_to_xyz(img: np.ndarray) -> np.ndarray:

  img = img.astype(np.float64)
  img = img / 255.0

  # Matriz CIE RGB -> XYZ. Sus filas son las ecuaciones de X, Y, Z
  # en función de (R, G, B), por lo que hay que aplicarla como
  # matrix @ [R, G, B] por píxel (de ahí el matrix.T al multiplicar
  # por la imagen vista como vectores fila).
  matrix = np.array([
      [0.490, 0.310, 0.200],
      [0.177, 0.813, 0.011],
      [.000, .010, 0.990],
  ])

  rgb_linear = _gamma_to_linear(img)

  xyz = rgb_linear @ matrix.T

  return xyz


def plot_XYZ(xyz_image: np.ndarray, figsize=(15, 5)):

    X = xyz_image[:, :, 0]
    Y = xyz_image[:, :, 1]
    Z = xyz_image[:, :, 2]

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    canales = [
        (X, "X", "Reds"),
        (Y, "Y (luminancia)", "gray"),
        (Z, "Z", "Blues"),
    ]

    for ax, (canal, titulo, cmap) in zip(axes, canales):
        im = ax.imshow(canal, cmap=cmap, vmin=canal.min(), vmax=canal.max())
        ax.set_title(titulo)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.tight_layout()

def rgb_to_lab(img: np.ndarray) -> np.ndarray:

  #Los parámetros X_n, Y_n, Z_n pueden variar
  _X_n = 1.0
  _Y_n = 1.0
  _Z_n = 1.0

  xyz = rgb_to_xyz(img)

  xr = xyz[:, :, 0] / _X_n
  yr = xyz[:, :, 1] / _Y_n
  zr = xyz[:, :, 2] / _Z_n

  delta = 6.0 / 29.0

  def f(t):
    return np.where(t > delta ** 3, np.cbrt(t), t / (3 * delta ** 2) + 4.0 / 29.0)

  fx, fy, fz = f(xr), f(yr), f(zr)

  L = 116.0 * fy - 16.0
  a = 500.0 * (fx - fy)
  b = 200.0 * (fy - fz)

  lab_img = np.stack([L, a, b], axis=-1)
  return lab_img

def plot_lab(lab_img: np.ndarray, figsize=(15, 5)):

    L = lab_img[:, :, 0]
    a = lab_img[:, :, 1]
    b = lab_img[:, :, 2]

    cmap_a = LinearSegmentedColormap.from_list("green_red", ["green", "whitesmoke", "red"])
    cmap_b = LinearSegmentedColormap.from_list("blue_yellow", ["blue", "whitesmoke", "yellow"])

    a_lim = np.abs(a).max() + 1e-10
    b_lim = np.abs(b).max() + 1e-10

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    canales = [
        (L, "L (Lightness)", "gray", 0, 100),
        (a, "a (verde ↔ rojo)", cmap_a, -a_lim, a_lim),
        (b, "b (azul ↔ amarillo)", cmap_b, -b_lim, b_lim),
    ]

    for ax, (canal, titulo, cmap, vmin, vmax) in zip(axes, canales):
        im = ax.imshow(canal, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(titulo)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.tight_layout()

In [ ]:
def rgb_to_lch(img: np.ndarray) -> np.ndarray:

    lab = rgb_to_lab(img)

    L = lab[:, :, 0]
    a = lab[:, :, 1]
    b = lab[:, :, 2]

    C = np.sqrt(a ** 2 + b ** 2)
    h = np.degrees(np.arctan2(b, a))
    h = np.where(h < 0, h + 360.0, h)

    lch_img = np.stack([L, C, h], axis=-1)
    return lch_img

def plot_lch(lch_img: np.ndarray, figsize=(15, 5)):

    L = lch_img[:, :, 0]
    C = lch_img[:, :, 1]
    h = lch_img[:, :, 2]

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    canales = [
        (L, "L (Lightness)", "gray", 0, 100),
        (C, "C (Chroma)", "magma", C.min(), C.max()),
        (h, "h (hue)", "hsv", 0, 360),
    ]

    for ax, (canal, titulo, cmap, vmin, vmax) in zip(axes, canales):
        im = ax.imshow(canal, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(titulo)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.tight_layout()

In [ ]:
lchP1= rgb_to_lch(P1BGR)
plot_lch(lchP1)

In [ ]:
def xyz_to_rgb(xyz_img: np.ndarray) -> np.ndarray:

    matrix = np.array([
        [0.490, 0.310, 0.200],
        [0.177, 0.813, 0.011],
        [.000, .010, 0.990],
    ])

    inv = np.linalg.inv(matrix)

    rgb_linear = xyz_img @ inv.T
    rgb_linear = np.clip(rgb_linear, 0.0, 1.0)

    #Aplicar corrección gamma
    rgb = rgb_linear ** (1.0 / 2.2)

    return np.clip(rgb, 0.0, 1.0)

def lab_to_xyz(lab_img: np.ndarray) -> np.ndarray:

    _X_n = 1.0
    _Y_n= 1.0
    _Z_n = 1.0

    L = lab_img[:, :, 0]
    a = lab_img[:, :, 1]
    b = lab_img[:, :, 2]

    fy = (L + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0

    delta = 6.0 / 29.0

    def f_inv(f):
        return np.where(f > delta, f ** 3, 3 * delta ** 2 * (f - 4.0 / 29.0))

    X = f_inv(fx) * _X_n
    Y = f_inv(fy) * _Y_n
    Z = f_inv(fz) * _Z_n

    return np.stack([X, Y, Z], axis=-1)

def lab_to_rgb(lab_img: np.ndarray) -> np.ndarray:
  return xyz_to_rgb(lab_to_xyz(lab_img))


def lch_to_lab(lch_img: np.ndarray) -> np.ndarray:
  """Inversa de rgb_to_lch: vuelve de coordenadas polares (C, h)
  a cartesianas (a, b)."""

  L = lch_img[:, :, 0]
  C = lch_img[:, :, 1]
  h = np.radians(lch_img[:, :, 2])

  a = C * np.cos(h)
  b = C * np.sin(h)

  return np.stack([L, a, b], axis=-1)


def lch_to_rgb(lch_img: np.ndarray) -> np.ndarray:
  return lab_to_rgb(lch_to_lab(lch_img))
def tune_lch(rgb_img: np.ndarray):
  #Análogo a tune_cylindrical pero para LCh*

  base = rgb_to_lch(rgb_img)

  def _update(delta_h=0.0, scale_c=1.0, scale_l=1.0):
    tuned = base.copy()
    tuned[:, :, 0] = np.clip(tuned[:, :, 0] * scale_l, 0.0, 100.0)
    tuned[:, :, 1] = np.clip(tuned[:, :, 1] * scale_c, 0.0, None)
    tuned[:, :, 2] = (tuned[:, :, 2] + delta_h) % 360.0

    rgb_out = lch_to_rgb(tuned)

    plt.figure(figsize=(6, 6))
    plt.imshow(rgb_out)
    plt.axis("off")
    plt.show()

  interact(
      _update,
      delta_h=FloatSlider(min=-180, max=180, step=5, value=0.0, description="Delta h"),
      scale_c=FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0, description="Escala C"),
      scale_l=FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0, description="Escala L*"),
  )


In [ ]:
tune_lch(P1BGR)